In [ ]:
import utils
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from scipy.stats import pearsonr
import matplotlib.pyplot as plt


from torch.nn.utils.rnn import pad_packed_sequence
from torch.utils.data import DataLoader

In [ ]:
# Load the pickle files
# scaled_train = pd.read_pickle("scaled_train_with_modifiedNIA_0725.pkl")
# scaled_test = pd.read_pickle("scaled_test_with_modifiedNIA_0725.pkl")
total_data = pd.read_csv('inputs/livings.csv')
# trying remove the four pathologies columns to see is this good.
# total_data = total_data.drop(columns=['gpath', 'tangles', 'amyloid', 'niareagansc'])
print(f"The shape of the total livings data is {total_data.shape}") # Previous one (20434, 65)
print(f"The number of unique `Projid` is {total_data['projid'].nunique()} under only livings data.")
total_data.head()


In [ ]:
# Checking data
na_counts = total_data.isna().sum()
na_columns = na_counts[na_counts > 0]
print(na_columns) # It seems I need to remove all the rows with any NA in it except for the 4 pathologies rows.

In [ ]:
# In the previous version, there is no NA. Then I thought maybe the NAs are the problem.
# The below code is used to remove all the rows including NA except the NA in the 5 columns.

cols_to_ignore = ['gpath', 'tangles', 'amyloid', 'niareagansc', 'cogdx']

cols_to_check = [col for col in total_data.columns if col not in cols_to_ignore]

total_data_clean = total_data.dropna(subset=cols_to_check)

print(total_data_clean.shape)


In [ ]:
# Checking data
na_counts_clean = total_data_clean.isna().sum()
na_columns_clean = na_counts_clean[na_counts_clean > 0]
print(na_columns_clean) # It seems I need to remove all the rows with any NA in it except for the 4 pathologies rows.

In [ ]:
# Change the name of 'total_data_clean' back to 'total_data' for the easier following steps.
total_data = total_data_clean

In [ ]:
class Model_AllTimeSteps_BiLSTMasTrue(nn.Module): # Original Name as LSTMModelAllTimeSteps
    """
    BiLSTM model to predict outcomes at ALL time steps.
    Same architecture/weights as your original last-step model,
    without the extra ReLU layer.
    """
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers,
                 dropout=0.5, bidirectional=True):
        super().__init__()
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1

        self.lstm = nn.LSTM(
            input_dim,
            hidden_dim,
            num_layers,
            dropout=(dropout if num_layers > 1 else 0.0),
            batch_first=True,
            bidirectional=bidirectional
        )

        # BatchNorm and FC must match hidden_dim * num_directions
        self.batch_norm = nn.BatchNorm1d(hidden_dim * self.num_directions)
        self.fc = nn.Linear(hidden_dim * self.num_directions, output_dim)

    def forward(self, x):
        # x: PackedSequence
        packed_out, _ = self.lstm(x)
        padded_out, lengths = pad_packed_sequence(packed_out, batch_first=True)
        # padded_out shape: (batch, seq_len, hidden_dim * num_directions)

        # BatchNorm over feature dim: reshape to (batch, features, seq_len)
        normed = self.batch_norm(padded_out.transpose(1, 2))
        normed = normed.transpose(1, 2)  # back to (batch, seq_len, hidden_dim * num_directions)

        # Linear projection at each time step
        preds = self.fc(normed)  # (batch, seq_len, output_dim)
        return preds


In [ ]:
class Model_AllTimeSteps_BiLSTMasTrue_ReLU(nn.Module): # Original Name as LSTMModelAllTimeSteps_BiLSTMasTrue_ReLU
    """
    Modified LSTM model to predict outcomes at ALL time steps.
    Uses the same architecture/weights as your original last-step model.
    """
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers, dropout=0.5, bidirectional=True):
        super().__init__()
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1
        
        self.lstm = nn.LSTM(
            input_dim,
            hidden_dim,
            num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
            bidirectional=bidirectional
        )
        
        # Adjust BatchNorm and FC input dimensions based on bidirectionality
        self.batch_norm = nn.BatchNorm1d(hidden_dim * self.num_directions)
        self.relu = nn.ReLU()  
        self.fc = nn.Linear(hidden_dim * self.num_directions, output_dim)


    def forward(self, x):
        # x is the input in PackedSequence format

        # 1. Pass through LSTM to get the outputs for all time steps (in packed format)
        packed_output, _ = self.lstm(x)
        
        # 2. Convert packed_output to a padded tensor and also obtain the actual length of each sequence
        padded_output, lengths = pad_packed_sequence(packed_output, batch_first=True)
        # padded_output 的形状: (batch_size, max_seq_length, hidden_dim * num_directions)
        
        # 3. Apply BatchNorm1d: first reshape the tensor to (batch_size, features, seq_length)
        norm_output = self.batch_norm(padded_output.transpose(1, 2))
        norm_output = norm_output.transpose(1, 2)  # 转回 (batch_size, seq_length, hidden_dim * num_directions)
        
        # 4. Apply ReLU activation function
        relu_output = self.relu(norm_output)
        
        # 5. Apply the fully connected layer to all time steps to obtain the final predictions
        predictions = self.fc(relu_output)  # 输出形状: (batch_size, seq_length, output_dim)
        
        return predictions


In [ ]:
# Create sequences while preserving projid information
def create_sequences_with_ids(df, ids, feature_cols, target_cols):
    sequences = []
    for projid in ids:
        patient_df = df[df['projid'] == projid].sort_values('fu_year')
        features = patient_df[feature_cols].values.astype('float32')
        targets = patient_df[target_cols].values.astype('float32')
        sequences.append((projid, features, targets))
    return sequences


# Custom collate function to handle projids
def custom_collate_with_ids(batch):
    projids = [item[0] for item in batch]
    features = [torch.FloatTensor(item[1]) for item in batch]
    targets = [torch.FloatTensor(item[2]) for item in batch]
    
    # Sort sequences by length (descending) for packing
    lengths = [f.shape[0] for f in features]
    sort_idx = sorted(range(len(lengths)), key=lambda k: lengths[k], reverse=True)
    
    projids = [projids[i] for i in sort_idx]
    features = [features[i] for i in sort_idx]
    targets = [targets[i] for i in sort_idx]
    
    # Pack sequences
    packed_features = torch.nn.utils.rnn.pack_sequence(features, enforce_sorted=False)
    packed_targets = torch.nn.utils.rnn.pack_sequence(targets, enforce_sorted=False)
    
    return projids, packed_features, packed_targets

In [ ]:
# Generate sequences for total data
feature_columns = total_data.drop(columns=['projid', 'study', 'fu_year', 'cogdx',
                                            'amyloid', 'gpath', 'tangles', 'niareagansc']).columns.tolist()
target_columns = ['gpath', 'tangles', 'amyloid', 'niareagansc']

ids = total_data.projid.unique()
# Create sequences with projids
sequences = create_sequences_with_ids(total_data, ids, feature_columns, target_columns)

# Create DataLoader with projid tracking
data_loader = DataLoader(utils.VariableLengthTensorDataset(sequences),
                         batch_size=len(sequences),
                         collate_fn=custom_collate_with_ids,
                         shuffle=False)

Understanding the meaning of the sequence features.

In [ ]:
# # 假设 sequences 是由 create_sequences_with_ids 生成的 (projid, feature, target) 元组列表
print("First sequence feature shape:", sequences[0][1].shape[0])
print("Second sequence feature shape:", sequences[1][1].shape)
print("Third sequence feature shape:", sequences[2][1].shape)
print('')
# print("First sequence targets shape:", sequences[0][2].shape)
# print("Second sequence targets shape:", sequences[1][2].shape)
# print("Third sequence targets shape:", sequences[2][2].shape)

# Get prediction for gpath (LSTMReLU)

In [ ]:
# Load your pretrained weights into the modified model
model = Model_AllTimeSteps_BiLSTMasTrue_ReLU(
    input_dim=57, 
    hidden_dim=8, 
    output_dim=3, 
    num_layers=4, 
    dropout=0,
    bidirectional=False
)
model.load_state_dict(torch.load("inputs/final_model_gpath_LSTMReLU.pth"))  # Reuse your trained weights

model.eval()
# During inference
results = []
with torch.no_grad():
    for batch_projids, batch_x, batch_y in data_loader: #batch_y
        # Get predictions from model (shape: batch_size, seq_len, 4)
        predictions = model(batch_x)
        # Get original sequence lengths from the packed input
        _, lengths = torch.nn.utils.rnn.pad_packed_sequence(batch_x, batch_first=True)
        # Convert predictions to numpy
        padded_preds = predictions.numpy()
        
        # Iterate through each sample in the batch
        for i, (projid, seq_len) in enumerate(zip(batch_projids, lengths)):
            # Get corresponding predictions and fu_years
            preds = padded_preds[i, :seq_len]  # Slice to actual sequence length
            fu_years = total_data[total_data['projid'] == projid]['fu_year'].sort_values().values
            
            # Store results
            for t in range(seq_len):
                results.append({
                    'projid': projid,
                    'fu_year': fu_years[t],
                    'gpath_pred': preds[t, 0],
                    'tangles_pred': preds[t, 1],
                    # 'amyloid_pred': preds[t, 2],
                    'niareagansc_pred': preds[t, 2]
                })

# Convert to DataFrame
results_df = pd.DataFrame(results)

In [ ]:
results_df[['projid', 'fu_year', 'gpath_pred']]

In [ ]:
gpath_df = results_df[['projid', 'fu_year', 'gpath_pred']]
print(gpath_df)

- validate the prediction R^2 for the last visit
- After adding patients with ADD = 0, this method to plot is useless.
- Because for these added patients there are no value for true pathologies and only have values for the imputed pathologies.

# Get prediction for tangles (LSTM)

In [ ]:
# Load your pretrained weights into the modified model
model = Model_AllTimeSteps_BiLSTMasTrue(
    input_dim=57, 
    hidden_dim=16, 
    output_dim=3, 
    num_layers=3, 
    dropout=0,
    bidirectional=False
)
model.load_state_dict(torch.load("inputs/final_model_tangles_LSTM.pth"))  # Reuse your trained weights

model.eval()
# During inference
results = []
with torch.no_grad():
    for batch_projids, batch_x, batch_y in data_loader:
        # Get predictions from model (shape: batch_size, seq_len, 4)
        predictions = model(batch_x)
        # Get original sequence lengths from the packed input
        _, lengths = torch.nn.utils.rnn.pad_packed_sequence(batch_x, batch_first=True)
        # Convert predictions to numpy
        padded_preds = predictions.numpy()
        
        # Iterate through each sample in the batch
        for i, (projid, seq_len) in enumerate(zip(batch_projids, lengths)):
            # Get corresponding predictions and fu_years
            preds = padded_preds[i, :seq_len]  # Slice to actual sequence length
            fu_years = total_data[total_data['projid'] == projid]['fu_year'].sort_values().values
            
            # Store results
            for t in range(seq_len):
                results.append({
                    'projid': projid,
                    'fu_year': fu_years[t],
                    # 'gpath_pred': preds[t, 0],
                    'amyloid_pred': preds[t, 0],
                    'tangles_pred': preds[t, 1],
                    'niareagansc_pred': preds[t, 2]               
                    })

# Convert to DataFrame
results_df = pd.DataFrame(results)

In [ ]:
tangles_df = results_df[['projid', 'fu_year', 'tangles_pred']]
print(tangles_df)

- validate the prediction R^2 for the last visit

In [ ]:

# Get last visit predictions for each subject
last_visit_preds = results_df.sort_values(['projid', 'fu_year']).groupby('projid').tail(1)

# Merge with original data
merged_df = pd.merge(
    last_visit_preds,
    total_data[['projid', 'fu_year', 'gpath', 'tangles', 'amyloid', 'niareagansc']],
    on=['projid', 'fu_year']
)

# Calculate R^2
target = 'tangles'
metrics = []

y_true = merged_df[f'{target}']
y_pred = merged_df[f'{target}_pred']

r, _ = pearsonr(y_true, y_pred)
r2_pearson = r**2

print(f"Pearson R²: {r2_pearson:.3f}")
print(f"Sample Size: {len(y_true)}")

In [ ]:
# Scatter plot of actual vs predicted values
plt.scatter(y_true, y_pred)
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.title("Actual vs Predicted tangles")

plt.show()

# Get prediction for amyloid (BiLSTM)

In [ ]:
# Load your pretrained weights into the modified model
model = Model_AllTimeSteps_BiLSTMasTrue(
    input_dim=57, 
    hidden_dim=32, 
    output_dim=4, 
    num_layers=3, 
    dropout=0.2
)
model.load_state_dict(torch.load("inputs/final_model_amyloid_BiLSTM.pth"))  # Reuse your trained weights

model.eval()
# During inference
results = []
with torch.no_grad():
    for batch_projids, batch_x, batch_y in data_loader:
        # Get predictions from model (shape: batch_size, seq_len, 4)
        predictions = model(batch_x)
        # Get original sequence lengths from the packed input
        _, lengths = torch.nn.utils.rnn.pad_packed_sequence(batch_x, batch_first=True)
        # Convert predictions to numpy
        padded_preds = predictions.numpy()
        
        # Iterate through each sample in the batch
        for i, (projid, seq_len) in enumerate(zip(batch_projids, lengths)):
            # Get corresponding predictions and fu_years
            preds = padded_preds[i, :seq_len]  # Slice to actual sequence length
            fu_years = total_data[total_data['projid'] == projid]['fu_year'].sort_values().values
            
            # Store results
            for t in range(seq_len):
                results.append({
                    'projid': projid,
                    'fu_year': fu_years[t],
                    'gpath_pred': preds[t, 0],
                    'tangles_pred': preds[t, 1],
                    'amyloid_pred': preds[t, 2],
                    'niareagansc_pred': preds[t, 3]
                })

# Convert to DataFrame
results_df = pd.DataFrame(results)

In [ ]:
amyloid_df = results_df[['projid', 'fu_year', 'amyloid_pred']]
print(amyloid_df)

- validate the prediction R^2 for the last visit

In [ ]:
# Get last visit predictions for each subject
last_visit_preds = results_df.sort_values(['projid', 'fu_year']).groupby('projid').tail(1)

# Merge with original data
merged_df = pd.merge(
    last_visit_preds,
    total_data[['projid', 'fu_year', 'gpath', 'tangles', 'amyloid', 'niareagansc']],
    on=['projid', 'fu_year']
)

# Calculate R^2
target = 'tangles'
metrics = []

y_true = merged_df[f'{target}']
y_pred = merged_df[f'{target}_pred']

r, _ = pearsonr(y_true, y_pred)
r2_pearson = r**2

print(f"Pearson R²: {r2_pearson:.3f}")
print(f"Sample Size: {len(y_true)}")

In [ ]:
# Scatter plot of actual vs predicted values
plt.scatter(y_true, y_pred)
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.title("Actual vs Predicted amyloid")

plt.show()

# Get prediction for nia (BiLSTM)

In [ ]:
# Load your pretrained weights into the modified model
model = Model_AllTimeSteps_BiLSTMasTrue(
    input_dim=57, 
    hidden_dim=32, 
    output_dim=4, 
    num_layers=4, 
    dropout=0
)
model.load_state_dict(torch.load("inputs/final_model_nia_BiLSTM.pth"))  # Reuse your trained weights

model.eval()
# During inference
results = []
with torch.no_grad():
    for batch_projids, batch_x, batch_y in data_loader:
        # Get predictions from model (shape: batch_size, seq_len, 4)
        predictions = model(batch_x)
        # Get original sequence lengths from the packed input
        _, lengths = torch.nn.utils.rnn.pad_packed_sequence(batch_x, batch_first=True)
        # Convert predictions to numpy
        padded_preds = predictions.numpy()
        
        # Iterate through each sample in the batch
        for i, (projid, seq_len) in enumerate(zip(batch_projids, lengths)):
            # Get corresponding predictions and fu_years
            preds = padded_preds[i, :seq_len]  # Slice to actual sequence length
            fu_years = total_data[total_data['projid'] == projid]['fu_year'].sort_values().values
            
            # Store results
            for t in range(seq_len):
                results.append({
                    'projid': projid,
                    'fu_year': fu_years[t],
                    'gpath_pred': preds[t, 0],
                    'tangles_pred': preds[t, 1],
                    'amyloid_pred': preds[t, 2],
                    'niareagansc_pred': preds[t, 3]
                })

# Convert to DataFrame
results_df = pd.DataFrame(results)

In [ ]:
nia_df = results_df[['projid', 'fu_year', 'niareagansc_pred']]
print(nia_df)

In [ ]:
# Get last visit predictions for each subject
last_visit_preds = results_df.sort_values(['projid', 'fu_year']).groupby('projid').tail(1)

# Merge with original data
merged_df = pd.merge(
    last_visit_preds,
    total_data[['projid', 'fu_year', 'gpath', 'tangles', 'amyloid', 'niareagansc']],
    on=['projid', 'fu_year']
)

# Calculate R^2
target = 'niareagansc'
metrics = []

y_true = merged_df[f'{target}']
y_pred = merged_df[f'{target}_pred']

r, _ = pearsonr(y_true, y_pred)
r2_pearson = r**2

print(f"Pearson R²: {r2_pearson:.3f}")
print(f"Sample Size: {len(y_true)}")

In [ ]:
# Scatter plot of actual vs predicted values
plt.scatter(y_true, y_pred)
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.title("Actual vs Predicted nia")

plt.show()

# Save predictions

In [ ]:
merged_df = pd.merge(gpath_df, tangles_df, on=['projid', 'fu_year'])
merged_df = pd.merge(merged_df, amyloid_df, on=['projid', 'fu_year'])
merged_results_df = pd.merge(merged_df, nia_df, on=['projid', 'fu_year'])

desired_columns = ['projid', 'fu_year', 'gpath_pred', 'tangles_pred', 'amyloid_pred', 'niareagansc_pred']
merged_results_df = merged_results_df[desired_columns]
print(merged_results_df)


In [ ]:
merged_results_df.shape

In [ ]:
merged_results_df.to_csv('outputs/predicted_gpath_tangles_amyloid_nia_ModifiedNIAValue_only_for_livings1106.csv',
                         index=None,
                         encoding='utf-8-sig' 
                         )